In [1]:
import os
import warnings
from langchain_core._api import LangChainDeprecationWarning

# 1. Set the USER_AGENT environment variable to silence scraping tool warnings
os.environ["USER_AGENT"] = "AccommodationMatcher/1.0"

# 2. Silence standard Python deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 3. Silence LangChain-specific warnings via string matching (no explicit imports needed)
warnings.filterwarnings("ignore", message=".*allowed_objects.*")
warnings.filterwarnings("ignore", message=".*langchain-community.*")

import operator
from typing import TypedDict, List, Dict, Annotated, Sequence

from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, ToolMessage

from langgraph.managed.is_last_step import RemainingSteps
from langgraph.prebuilt import create_react_agent

In [2]:
hotel_db = SQLDatabase.from_uri(
    'sqlite:///cornwall_hotels.db',
)
for table in hotel_db.get_usable_table_names():
    print(f'Table: {table}')
    res = hotel_db.run(f'SELECT * FROM {table} LIMIT 5;')
    print(res)
    print ()

Table: hotel_room_offers
[(1, 1, 5, 120.0, 180.0), (2, 2, 2, 95.0, 150.0), (3, 3, 8, 110.0, 170.0), (4, 4, 3, 130.0, 200.0), (5, 5, 4, 105.0, 160.0)]

Table: hotels
[(1, 'Seaview Hotel', 'Newquay', '1 Beach Rd, Newquay', 4.5, 'A beautiful hotel overlooking the sea.'), (2, 'Harbour Inn', 'Falmouth', '12 Harbour St, Falmouth', 4.2, 'Charming inn near the harbour.'), (3, 'Cornish Retreat', 'St Austell', '5 Retreat Ln, St Austell', 4.0, 'Relaxing retreat in the heart of Cornwall.'), (4, 'Penzance Palace', 'Penzance', '22 Promenade, Penzance', 4.7, 'Luxury hotel with sea views.'), (5, 'The Camborne Arms', 'Camborne', '8 Main St, Camborne', 4.1, 'Friendly hotel in Camborne.')]



In [3]:
from langchain_ollama import ChatOllama
LOCAL_LLM = 'gemma4:12b-mlx'
TEMPERATURE = 0.0

llm_model = ChatOllama(
    model=LOCAL_LLM,
    temperature=TEMPERATURE,
    use_responses_api=True
)

hotel_db_toolkit = SQLDatabaseToolkit(db=hotel_db, llm=llm_model)
hotel_db_toolkit_tools = hotel_db_toolkit.get_tools()

In [4]:
class BnBOffer(TypedDict):
    # Define the return type of the BnB availability tool
    bnb_id: int
    bnb_name: str
    town: str
    available_rooms: int
    price_per_room: float

class BnBBookingService:
    # Define the BnB availability tool
    @staticmethod
    # Call the BnB booking service to get the offers
    def get_offers_near_town(town: str, num_rooms: int) -> List[BnBOffer]:
        # Mocked REST API response: multiple BnBs per destination
        mock_bnb_offers = [ # Mocked BnB offers
            # Newquay
            {"bnb_id": 1, "bnb_name": "Seaside BnB", 
            "town": "Newquay", "available_rooms": 3, 
            "price_per_room": 80.0},
            {"bnb_id": 2, "bnb_name": "Surfside Guesthouse", 
            "town": "Newquay", "available_rooms": 2, 
            "price_per_room": 85.0},
            # Falmouth
            {"bnb_id": 3, "bnb_name": "Harbour View BnB", 
            "town": "Falmouth", "available_rooms": 4, 
            "price_per_room": 78.0},
            {"bnb_id": 4, "bnb_name": "Seafarer's Rest", 
            "town": "Falmouth", "available_rooms": 1, 
            "price_per_room": 90.0},
            # St Austell
            {"bnb_id": 5, "bnb_name": "Garden Gate BnB", 
            "town": "St Austell", "available_rooms": 2, "price_per_room": 82.0},
            {"bnb_id": 6, "bnb_name": "Coastal Cottage BnB", 
            "town": "St Austell", "available_rooms": 3, "price_per_room": 88.0},
            # Penzance
            {"bnb_id": 7, "bnb_name": "Penzance Pier BnB", 
            "town": "Penzance", "available_rooms": 2, "price_per_room": 95.0},
            {"bnb_id": 8, "bnb_name": "Cornish Charm BnB", 
            "town": "Penzance", "available_rooms": 3, "price_per_room": 87.0},
            # Camborne
            {"bnb_id": 9, "bnb_name": "Camborne Corner BnB", 
            "town": "Camborne", "available_rooms": 2, "price_per_room": 75.0},
            {"bnb_id": 10, "bnb_name": "Rose Cottage BnB", 
            "town": "Camborne", "available_rooms": 2, "price_per_room": 79.0},
            # Hayle
            {"bnb_id": 11, "bnb_name": "Hayle Haven BnB", 
            "town": "Hayle", "available_rooms": 3, "price_per_room": 83.0},
            {"bnb_id": 12, "bnb_name": "Dune View BnB", 
            "town": "Hayle", "available_rooms": 1, "price_per_room": 81.0},
            # Land's End
            {"bnb_id": 13, "bnb_name": "Land's End Lookout BnB", 
            "town": "Land's End", "available_rooms": 2, "price_per_room": 100.0},
            {"bnb_id": 14, "bnb_name": "Atlantic Edge BnB", 
            "town": "Land's End", "available_rooms": 2, "price_per_room": 105.0},
            # Bude
            {"bnb_id": 15, "bnb_name": "Bude Beach BnB", 
            "town": "Bude", "available_rooms": 2, "price_per_room": 77.0},
            {"bnb_id": 16, "bnb_name": "Cliffside BnB", 
            "town": "Bude", "available_rooms": 3, "price_per_room": 80.0},
            # Padstow
            {"bnb_id": 17, "bnb_name": "Padstow Harbour BnB", 
            "town": "Padstow", "available_rooms": 2, "price_per_room": 92.0},
            {"bnb_id": 18, "bnb_name": "Fisherman's Rest BnB", 
            "town": "Padstow", "available_rooms": 2, "price_per_room": 89.0},
            # St Ives
            {"bnb_id": 19, "bnb_name": "St Ives Bay BnB", "town": "St Ives", "available_rooms": 3, "price_per_room": 97.0},
            {"bnb_id": 20, "bnb_name": "Artists' Retreat BnB", "town": "St Ives", "available_rooms": 2, "price_per_room": 102.0},
            # Looe
            {"bnb_id": 21, "bnb_name": "Looe Riverside BnB", "town": "Looe", "available_rooms": 2, "price_per_room": 84.0},
            {"bnb_id": 22, "bnb_name": "Harbour Lights BnB", "town": "Looe", "available_rooms": 2, "price_per_room": 86.0},
            # Polperro
            {"bnb_id": 23, "bnb_name": "Polperro Cove BnB", "town": "Polperro", "available_rooms": 2, "price_per_room": 91.0},
            {"bnb_id": 24, "bnb_name": "Smuggler's Rest BnB", "town": "Polperro", "available_rooms": 2, "price_per_room": 93.0},
            # Mevagissey
            {"bnb_id": 25, "bnb_name": "Mevagissey Harbour BnB", "town": "Mevagissey", "available_rooms": 2, "price_per_room": 90.0},
            {"bnb_id": 26, "bnb_name": "Seafarer's BnB", "town": "Mevagissey", "available_rooms": 2, "price_per_room": 88.0},
            # Port Isaac
            {"bnb_id": 27, "bnb_name": "Port Isaac View BnB", 
            "town": "Port Isaac", "available_rooms": 2, 
            "price_per_room": 99.0},
            {"bnb_id": 28, "bnb_name": "Fisherman's Cottage BnB", 
            "town": "Port Isaac", "available_rooms": 2, 
            "price_per_room": 101.0},
            # Fowey
            {"bnb_id": 29, "bnb_name": "Fowey Quay BnB", 
            "town": "Fowey", "available_rooms": 2, 
            "price_per_room": 94.0},
            {"bnb_id": 30, "bnb_name": "Riverside Rest BnB", 
            "town": "Fowey", "available_rooms": 2, 
            "price_per_room": 96.0},
        ]
        offers = [
            offer for offer in mock_bnb_offers 
            if offer["town"].lower() == town.lower()
            and offer["available_rooms"] >= num_rooms
        ]
        return offers

In [5]:
@tool(description='''Check BnB room availability and price for a destination in Cornwall.''')
def check_bnb_availability(destination_town: str, num_rooms: int) -> List[Dict]:
    """Check BnB room availability and price for a specific town.
    
    Args:
        destination_town: The specific town name (e.g., 'Newquay', 'Falmouth', 'St Ives'). 
        num_rooms: The number of required rooms.
    """
    return BnBBookingService.get_offers_near_town(destination_town, num_rooms)    

In [6]:
booking_tools = hotel_db_toolkit_tools + [check_bnb_availability]

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps    

In [7]:
def strict_orchestration_modifier(state: AgentState) -> list:
    """Appends strict forcing rules to the context stack EXACTLY once."""
    base_instructions = (
        "\n\n[SYSTEM CONSTRAINT: You are a dual-engine holiday accommodation matcher.\n"
        "CRITICAL FORCING RULES:\n"
        "1. To answer completely, you MUST search for BOTH hotels (using sql_db_query) "
        "AND bed & breakfasts (using 'check_bnb_availability').\n"
        "2. Do NOT stop or reply after running a SQL query. You must immediately call "
        "'check_bnb_availability' for the same town before writing your final response.\n"
        "3. Output a single combined Markdown table containing BOTH types of accommodations.]"
    )
    
    messages = list(state["messages"])
    
    # Check if we have already injected the system instructions to prevent infinite duplicates
    has_instructions = any(
        isinstance(m, SystemMessage) and "dual-engine holiday accommodation matcher" in m.content 
        for m in messages
    )
    
    # Only append if it's not already in the history
    if messages and not has_instructions:
        messages.append(SystemMessage(content=base_instructions))
        
    return messages

accommodation_booking_agent = create_react_agent(
    model=llm_model,
    tools=booking_tools,
    state_schema=AgentState,
    prompt=strict_orchestration_modifier
)

In [8]:
def chat_loop():
    print("Booking Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        
        # --- FIX HERE: Add a recursion limit config ---
        # 10 steps is more than enough for a fallback batch
        config = {"recursion_limit": 10} 
        
        try:
            result = accommodation_booking_agent.invoke(state, config=config)
            print("\n\n\n")
            print(result)
            print("\n\n\n")
            response_msg = result["messages"][-1].content
            print(f"Assistant: {response_msg}\n")
        except GraphRecursionError:
            # Catch the limit gracefully if it hits a runaway loop
            print("\nAssistant: I'm sorry, I couldn't find any accomodation.\n")

            

In [9]:
chat_loop()

Booking Assistant (type 'exit' to quit)


You:  Are there any hotel or BnB's available in Penzance for 1 person?






{'messages': [HumanMessage(content="Are there any hotel or BnB's available in Penzance for 1 person?", additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-mlx', 'created_at': '2026-06-11T23:47:27.957764Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7143084084, 'load_duration': 1737658667, 'prompt_eval_count': 629, 'prompt_eval_duration': 1051163084, 'eval_count': 127, 'eval_duration': 4351653750, 'logprobs': None, 'model_name': 'gemma4:12b-mlx', 'model_provider': 'ollama'}, id='lc_run--019eb915-4a6c-72f2-9035-2e1de6aba2a8-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'b4aa7222-ef21-4d12-855f-3e9475c5c947', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 629, 'output_tokens': 127, 'total_tokens': 756}), ToolMessage(content='hotel_room_offers, hotels', name='sql_db_list_tables', tool_call_id='b4aa7222-ef21-4d12-855f-3e9475c5c947'), 

You:  exit
